# Cross-condition results explorer

Aggregate every scored run (`runs/*/recall.csv`) into one view — the summary test
(D vs C0) and the representation test (D vs PDF-outline) side by side.

**Kernel:** `pandas` + `altair` (`.venv`). Read-only; each run's `recall.csv` is
produced by `scripts/score_recall.py` (the single source of the metric). Color encodes
**baseline vs variant** — the validated 2-hue brand pair — with higher-cardinality
dimensions faceted, never given invented colors.

In [ ]:
from pathlib import Path
import sys, glob, csv, os

def find_repo(start=Path.cwd()):
    for p in [start, *start.parents]:
        if (p / "runs").exists() and (p / "scripts" / "score_recall.py").exists():
            return p
    raise RuntimeError("run this notebook from inside the repo")

REPO = find_repo()
sys.path.insert(0, str(REPO / "scripts"))
import pandas as pd
import altair as alt
import viz_theme
viz_theme.register()
P = viz_theme.PALETTE
ROLE = alt.Scale(domain=["baseline", "variant"], range=[P["cobalt"], P["primary"]])

MIN_QIDS = 5          # ignore smoke runs; raise to focus on full studies

def short(ix):
    return ix.replace("IDX-", "").replace("-rfc9110", "")

# Each run needs a recall.csv (produce it with scripts/score_recall.py). We read
# those directly — score_recall.py stays the single source of the metric.
rows = []
for rc in sorted(glob.glob(str(REPO / "runs" / "*" / "recall.csv"))):
    run = os.path.basename(os.path.dirname(rc))
    data = list(csv.DictReader(open(rc)))
    if len({r["qid"] for r in data}) < MIN_QIDS:
        continue
    condition = "+".join(short(i) for i in sorted({r["index"] for r in data}))
    for r in data:
        rows.append(dict(
            run=run, condition=condition, index=short(r["index"]), qid=r["qid"],
            category=r["category"], absence=r["absence"] == "True",
            recall=float(r["recall"]) if r["recall"] else None,
            content_tok=int(r["content_tok"] or 0)))
df = pd.DataFrame(rows)

# baseline = the index shared across the most conditions (the common reference);
# every other index is the "variant" being tested against it.
baseline = df.groupby("index").condition.nunique().idxmax()
df["role"] = df["index"].map(lambda i: "baseline" if i == baseline else "variant")
ans = df[~df.absence & df.recall.notna()]      # answerable questions only

print(f"baseline = {baseline}   conditions: {sorted(df.condition.unique())}")
print(f"indexes: {sorted(df['index'].unique())}   ({df.run.nunique()} runs, "
      f"{ans.qid.nunique()} questions)")

## Overview — recall by category x index

A sequential heatmap (one hue): darker = more
of the answer's gold sections were fetched. Scan columns to compare indexes per category.

In [ ]:
# recall by category x index (sequential = one hue; IDX-D averaged across its runs)
h = ans.groupby(["category", "index"], as_index=False).recall.mean()
alt.Chart(h).mark_rect(stroke=P["ground"], strokeWidth=2).encode(
    x=alt.X("index:N", title=None),
    y=alt.Y("category:N", title=None),
    color=alt.Color("recall:Q", title="recall",
                    scale=alt.Scale(range=viz_theme.SEQUENTIAL, domain=[0, 1])),
    tooltip=["category", "index", alt.Tooltip("recall:Q", format=".2f")]
).properties(title="Recall by category x index", width=240, height=200)

## The efficiency frontier

Recall is only half the story — pair it with the content
tokens spent to get it. A point up-and-left (high recall, few tokens) is the efficient one.

In [ ]:
# the efficiency frontier: recall vs the content tokens spent to get it.
eff = ans.groupby(["index", "role"], as_index=False).agg(
    recall=("recall", "mean"), content_tok=("content_tok", "mean"))
pts = alt.Chart(eff).mark_point(size=200, filled=True).encode(
    x=alt.X("content_tok:Q", title="mean content tokens / question"),
    y=alt.Y("recall:Q", title="mean recall", scale=alt.Scale(domain=[0, 1])),
    color=alt.Color("role:N", scale=ROLE, title="role"),
    tooltip=["index", alt.Tooltip("recall:Q", format=".3f"),
             alt.Tooltip("content_tok:Q", format=".0f")])
labels = pts.mark_text(dy=-12, color=P["ink"]).encode(text="index:N")
(pts + labels).properties(
    title="Efficiency frontier: recall vs content tokens", width=440, height=280)

## Where each variant helps or hurts

Baseline (cobalt) vs variant (burnt orange) per
category, one row per condition. Ties everywhere except a few categories = a *specific*
effect, not a global verdict.

In [ ]:
# where each variant helps or hurts vs the baseline, per category (small multiples)
g = ans.groupby(["condition", "category", "role"], as_index=False).recall.mean()
alt.Chart(g).mark_bar().encode(
    x=alt.X("role:N", title=None, axis=alt.Axis(labels=False, ticks=False)),
    y=alt.Y("recall:Q", scale=alt.Scale(domain=[0, 1]), title="recall"),
    color=alt.Color("role:N", scale=ROLE, title="role"),
    column=alt.Column("category:N", title=None,
                      header=alt.Header(labelOrient="bottom", labelAngle=-30)),
    row=alt.Row("condition:N", title=None),
    tooltip=["condition", "category", "role", alt.Tooltip("recall:Q", format=".2f")]
).properties(width=64, height=120)

## Biggest gaps (drill-down targets)

The cells to open in `2_analyze_one_run.ipynb`
and read the actual answers — remember recall@fetch under-credits coarse page-addressed arms.

In [ ]:
# biggest baseline-vs-variant recall gaps (the story hides in specific cells)
piv = (ans.pivot_table(index=["condition", "category"], columns="role",
                       values="recall", aggfunc="mean")
          .assign(gap=lambda d: d["baseline"] - d["variant"]))
piv.reindex(piv["gap"].abs().sort_values(ascending=False).index).round(3)